In [1]:
import pandas as pd

In [2]:
data = pd.read_csv(r'C:\Users\HP\Desktop\Projects - Data Analysis\RSF\ckm_raw_ehr_v4.csv')
data

,patient_id,date_of_birth,encounter_date,last_followup_date,age,sex,nationality_group,smoking_status,facility_type,encounter_type,...,icd_t2dm,icd_hypertension,icd_ckd,icd_dyslipidaemia,icd_obesity,icd_cvd_heart_failure,icd_cvd_ihd,icd_cvd_stroke,icd_cvd_pad,icd_subclinical_cvd
0,PT00001,1966-01-01,2020-02-28,2026-03-03,54,Male,Arab Expat,Never,Polyclinic,Outpatient,...,0,0,0,0,0,0,0,0,0,0
1,PT00001,1966-01-01,2021-11-08,2026-03-03,54,Male,Arab Expat,Never,Tertiary Hospital,Outpatient,...,0,0,0,0,0,0,0,0,0,0
2,PT00001,1966-01-01,2024-01-25,2026-03-03,54,Male,Arab Expat,Never,Polyclinic,Outpatient,...,0,0,0,0,0,0,0,0,0,0
3,PT00001,1966-01-01,2024-11-12,2026-03-03,54,Male,Arab Expat,Never,Primary Care,Inpatient,...,0,0,0,0,0,0,0,0,0,0
4,PT00001,1966-01-01,2025-04-19,2026-03-03,54,Male,Arab Expat,Never,Primary Care,Emergency,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10034,PT01999,1975-01-01,2026-06-05,2026-06-05,45,Male,Arab Expat,Former,Tertiary Hospital,Inpatient,...,0,0,0,0,1,0,0,0,0,0
10035,PT02000,1983-01-01,2020-01-26,2024-08-26,37,Female,South Asian,Former,Tertiary Hospital,Emergency,...,1,0,1,0,0,0,0,0,0,0
10036,PT02000,1983-01-01,2020-12-19,2024-08-26,37,Female,South Asian,Former,Primary Care,Outpatient,...,1,0,1,0,0,0,0,0,0,0
10037,PT02000,1983-01-01,2022-08-22,2024-08-26,37,Female,South Asian,Former,Polyclinic,Outpatient,...,1,0,1,0,1,0,0,0,0,0


Prevalence & Binary Logistic Regression

In [3]:
data['encounter_date'] = pd.to_datetime(data['encounter_date'])
data_sorted = data.sort_values(by='encounter_date')
data_sorted

,patient_id,date_of_birth,encounter_date,last_followup_date,age,sex,nationality_group,smoking_status,facility_type,encounter_type,...,icd_t2dm,icd_hypertension,icd_ckd,icd_dyslipidaemia,icd_obesity,icd_cvd_heart_failure,icd_cvd_ihd,icd_cvd_stroke,icd_cvd_pad,icd_subclinical_cvd
7746,PT01538,1980-01-01,2020-01-01,2025-05-28,40,Male,South Asian,Former,Tertiary Hospital,Outpatient,...,0,0,0,0,0,0,0,0,0,0
6100,PT01216,1955-01-01,2020-01-01,2024-03-04,65,Male,South Asian,Former,Tertiary Hospital,Outpatient,...,1,0,0,0,1,0,0,0,0,0
9257,PT01841,1974-01-01,2020-01-02,2025-07-16,46,Male,South Asian,Never,Primary Care,Outpatient,...,0,0,0,0,0,0,0,0,0,0
7897,PT01567,1972-01-01,2020-01-02,2026-06-09,48,Female,South Asian,Current,Tertiary Hospital,Outpatient,...,0,1,0,0,1,0,0,0,0,0
756,PT00158,1966-01-01,2020-01-02,2025-07-13,54,Female,South Asian,Never,Polyclinic,Outpatient,...,0,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,PT00157,1946-01-01,2026-06-28,2026-06-28,74,Female,South Asian,Never,Polyclinic,Outpatient,...,0,0,1,0,0,0,0,1,0,0
6742,PT01341,1970-01-01,2026-06-28,2026-06-28,50,Female,East Asian,Never,Primary Care,Emergency,...,0,0,0,0,0,0,0,0,0,0
1048,PT00215,1977-01-01,2026-06-29,2026-06-29,43,Female,East Asian,Never,Primary Care,Outpatient,...,1,0,0,0,0,0,0,0,0,0
2453,PT00497,1987-01-01,2026-06-29,2026-06-29,33,Female,Arab Expat,NaN,Polyclinic,Outpatient,...,0,1,0,0,1,0,1,0,0,0


In [4]:
grouped_data = data_sorted.groupby('patient_id').agg(
    encounter_date=('encounter_date', 'last'),
    age=('age', 'first'),
    sex=('sex', 'first'),
    nationality=('nationality_group', 'first'),
    waist=('waist_circumference_cm', 'last'),
    egfr=('egfr', 'last'), 
    uacr=('uacr_mg_g', 'max'),
    fasting_glucose=('fasting_glucose_mmol', 'last'),
    hba1c=('hba1c_pct', 'last'),
    bmi=('bmi', 'last'),
    ldl=('ldl_mmol', 'last'), 
    chol=('total_chol_mmol', 'last'),
    sbp=('sbp_mmhg', 'last'),
    dbp=('dbp_mmhg', 'last'),
    prediabetes=('icd_prediabetes', 'max'),
    t2dm=('icd_t2dm', 'max'),
    htn=('icd_hypertension', 'max'),
    ckd=('icd_ckd', 'max'),
    dyslipidemia=('icd_dyslipidaemia', 'max'),
    hf=('icd_cvd_heart_failure', 'max'),
    ihd=('icd_cvd_ihd', 'max'),
    stroke=('icd_cvd_stroke', 'max'),
    pad=('icd_cvd_pad', 'max'),
    subclinical=('icd_subclinical_cvd', 'max')
)
grouped_data

,encounter_date,age,sex,nationality,waist,egfr,uacr,fasting_glucose,hba1c,bmi,...,prediabetes,t2dm,htn,ckd,dyslipidemia,hf,ihd,stroke,pad,subclinical
patient_id,,,,,,,,,,,,,,,,,,,,,
PT00001,2026-03-03,54,Male,Arab Expat,96.7,123.9,19.2,5.45,6.0,20.8,...,0,0,0,0,0,0,0,0,0,0
PT00002,2025-03-19,46,Male,East Asian,91.0,62.0,242.3,6.16,5.6,35.8,...,0,0,1,1,0,0,1,0,0,0
PT00003,2025-12-05,57,Male,African,60.5,70.4,11.1,5.71,4.8,27.0,...,0,0,0,0,0,0,1,0,0,0
PT00004,2025-05-02,69,Male,Emirati,104.2,58.7,56.2,6.24,5.2,31.9,...,1,1,0,0,0,0,0,0,0,0
PT00005,2025-11-12,44,Female,South Asian,91.5,78.4,107.7,6.65,6.5,30.1,...,0,0,1,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PT01996,2023-09-13,62,Male,South Asian,NaN,78.8,64.0,3.67,7.0,27.4,...,0,0,0,1,0,0,0,0,0,0
PT01997,2025-06-08,47,Male,South Asian,93.6,69.4,111.0,5.12,7.4,33.4,...,0,0,0,0,0,0,0,0,0,0
PT01998,2026-05-31,35,Female,Emirati,NaN,86.9,64.1,NaN,4.4,30.5,...,0,0,0,0,0,0,0,0,0,0


In [5]:
encoded_data = grouped_data.copy()

for column in encoded_data.columns:
    if pd.api.types.is_string_dtype(encoded_data[column]):
        values = encoded_data[column].unique()
        print(f"For {column}: \nUnique values are: {values}")

For sex: 
Unique values are: <StringArray>
['Male', 'Female']
Length: 2, dtype: str
For nationality: 
Unique values are: <StringArray>
['Arab Expat', 'East Asian', 'African', 'Emirati', 'South Asian', 'Western']
Length: 6, dtype: str


In [6]:
encoded_data['sex'] = encoded_data['sex'].map({'Male': 1, 'Female': 2})
encoded_data['nationality'] = encoded_data['nationality'].map({
    'Arab Expat': 1, 'East Asian': 2, 'African': 3, 'Emirati': 4, 
    'South Asian': 5, 'Western': 6
})

In [7]:
encoded_data.to_csv('encoded_ckm_grouped_by_encounter_date.csv', index=False) 